In [1]:
import numpy as np
import pandas as pd
from scipy.special import spherical_jn
from scipy.integrate import cumulative_trapezoid

csv_file = "/root/geant4/fac/Kr_all_wavefunctions.csv"
df = pd.read_csv(csv_file)
df = df.dropna(subset=["r", "P", "Q"])

p = np.arange(0.0, 1000.0 + 0.05, 0.05)
Qgrid = np.arange(0.0, 100.0 + 0.05, 0.05)

print(df["Orbital"].unique())


def inspect_orbital(df, name, nrows=10):
    d = df[df["Orbital"] == name].copy().sort_values("r")

    print("\n==============================")
    print("Inspect:", name)
    print("==============================")
    print(d[["r", "P", "Q"]].head(nrows))
    print(d[["r", "P", "Q"]].tail(nrows))

    r = d["r"].to_numpy()
    P = d["P"].to_numpy()
    Qs = d["Q"].to_numpy()

    print("min r =", r.min(), "max r =", r.max(), "points =", len(r))
    print("raw max |P| =", np.max(np.abs(P)))
    print("raw max |Q| =", np.max(np.abs(Qs)))
    print("raw ∫(P²+Q²)dr =", np.trapezoid(P**2 + Qs**2, r))
    print("raw ∫r²(P²+Q²)dr =", np.trapezoid(r**2 * (P**2 + Qs**2), r))


def get_orbital(df, name):
    d = df[df["Orbital"] == name].copy().sort_values("r")

    r = d["r"].to_numpy()
    P = d["P"].to_numpy()
    Qs = d["Q"].to_numpy()

    norm = np.trapezoid(P**2 + Qs**2, r)

    P = P / np.sqrt(norm)
    Qs = Qs / np.sqrt(norm)

    print(name, "radial norm before =", norm)

    return r, P, Qs


def chi_transform(p_grid, r, radial, l, mode):
    out = []

    for pp in p_grid:
        jl = spherical_jn(l, pp*r)

        if mode == "no_r":
            integrand = radial * jl
        elif mode == "with_r":
            integrand = radial * jl * r
        elif mode == "with_r2":
            integrand = radial * jl * r**2
        else:
            raise ValueError("mode must be no_r, with_r, or with_r2")

        val = np.sqrt(2/np.pi) * np.trapezoid(integrand, r)
        out.append(val)

    return np.array(out)


def test_one_orbital(df, name, l, jtype):
    r, P, Qs = get_orbital(df, name)

    if jtype == "plus":
        l_small = l + 1
    elif jtype == "minus":
        l_small = l - 1
    else:
        raise ValueError("jtype must be plus or minus")

    print("\nMomentum-transform test for", name)

    for mode in ["no_r", "with_r", "with_r2"]:
        chiG = chi_transform(p, r, P, l, mode)
        chiF = chi_transform(p, r, Qs, l_small, mode)

        I = (chiG**2 + chiF**2) * p**2
        norm_I = np.trapezoid(I, p)

        integrand_J = (chiG**2 + chiF**2) * p
        rev = cumulative_trapezoid(
            integrand_J[::-1],
            p[::-1],
            initial=0
        )
        J_p = -0.5 * rev[::-1]
        J_Q = np.interp(Qgrid, p, J_p)

        norm_J = 2*np.trapezoid(J_Q, Qgrid)

        print(mode, "  ∫I(p)dp =", norm_I, "   2∫J(Q)dQ =", norm_J)


# =====================================================
# Print r, P, Q for some important orbitals
# =====================================================

inspect_orbital(df, "Kr_1s12")
inspect_orbital(df, "Kr_2s12")
inspect_orbital(df, "Kr_4p32")

# =====================================================
# Test which transform gives norm close to 1
# =====================================================

test_one_orbital(df, "Kr_1s12", 0, "plus")
test_one_orbital(df, "Kr_2s12", 0, "plus")
test_one_orbital(df, "Kr_2p12", 1, "minus")
test_one_orbital(df, "Kr_2p32", 1, "plus")
test_one_orbital(df, "Kr_4p32", 1, "plus")

['Kr_1s12' 'Kr_2p12' 'Kr_2p32' 'Kr_2s12' 'Kr_3d32' 'Kr_3d52' 'Kr_3p12'
 'Kr_3p32' 'Kr_3s12' 'Kr_4p12' 'Kr_4p32' 'Kr_4s12']

Inspect: Kr_1s12
               r         P             Q
2   2.777778e-08  0.000015 -5.468180e-10
3   3.046001e-08  0.000016 -6.574759e-10
4   3.340125e-08  0.000018 -7.905367e-10
5   3.662649e-08  0.000019 -9.505354e-10
6   4.016309e-08  0.000021 -1.142921e-09
7   4.404110e-08  0.000023 -1.374248e-09
8   4.829356e-08  0.000025 -1.652404e-09
9   5.295663e-08  0.000028 -1.986867e-09
10  5.806984e-08  0.000030 -2.389028e-09
11  6.367665e-08  0.000033 -2.872587e-09
            r             P             Q
200  0.524041  3.203361e-06 -3.930715e-07
201  0.552208  1.313572e-06 -1.605485e-07
202  0.581545  5.182847e-07 -6.306523e-08
203  0.612082  1.965789e-07 -2.379957e-08
204  0.643849  7.160500e-08 -8.619408e-09
205  0.676876  2.502456e-08 -2.992503e-09
206  0.711192  8.382625e-09 -9.948171e-10
207  0.746826  2.688760e-09 -3.162902e-10
208  0.783807  8.249819e-10 -1